# Fitting,A GP to a Simulated Neutral Cell

In this notebook, we fit a Gaussian process to a simulated neutral cell, to demonstrate recovery of the 300 s periodicity in the case where the recording is only 960 seconds long.

In [ ]:
# Autoreloading of modules and files without restarting notebook.
%reload_ext autoreload
%autoreload 2


import os
import pyro
import json
import torch
import pickle
import numpy as np
import pandas as pd
from quantities import s
import matplotlib.pyplot as plt
from matplotlib.axes import Axes


from rvm_analysis.colours import get_cell_colour
from rvm_analysis.converters import combine_neutral_and_save
from rvm_analysis.utils import get_cell_colour
from rvm_analysis.non_poisson_gpytorch_model import evaluate_model, compare_log_liks
from rvm_analysis.utils import bin_spikes_and_split
from rvm_analysis.non_poisson_gpytorch_model import nonlinearity, train_model
from rvm_analysis.non_poisson_gpytorch_model import create_mean_model,create_model, save_model_params, get_summary_parameters_and_fits
from rvm_analysis.non_poisson_gpytorch_model import set_mean_and_scale

from rvm_analysis.plotting import plot_rainclouds
from rvm_analysis.plotting import plot_interactive_raincloud



# Set random seed
torch.manual_seed(0)
pyro.set_rng_seed(0)
torch.set_default_dtype(torch.float64)

#! Force the backend back to matplotlib inline (after importing viziphant).
print(plt.get_backend())
%matplotlib inline


## Generate Simulated Data

In [ ]:
# Create a periodic signal similar to an ON/OFF cell.
def noisy_periodic_signal(x, period=300*s, N_per=40, N_aper=40):
    periodic_flag = np.zeros(N_per + N_aper)
    periodic_flag[:N_per] = 1

    periodic_component = ( np.sin(2 * np.pi * x / period)[:, np.newaxis] * 30 * periodic_flag)
    noise = np.random.normal( 0, 20, (x.shape[0], N_per + N_aper))
    sig = np.round(30 + periodic_component + noise, 0)

    return np.where(sig > 0, sig, 0)


# Generate training and test data
def generate_data(N_per = 40,N_aper=40, t_train = 660, T = 960 * s, bin_width = 5*s, plot=False):
    t = np.linspace(0*s,T,int(T / bin_width))
    y = noisy_periodic_signal(t,period=300*s,N_per=N_per, N_aper = N_aper)

    if plot:
        plt.plot(t,y)
        plt.show()

    x_train = t[np.where(t < t_train)][:,np.newaxis].squeeze()
    x_test = t[np.where(t >= t_train)][:,np.newaxis].squeeze()
    y_train = y[np.where(t< t_train)].T
    y_test = y[np.where(t >=t_train)].T

    print(y_test.shape)

    data = {"NEUTRAL": (np.array(y_train),np.array(y_test),x_train.flatten(),x_test.flatten())}

    return data

In [ ]:
## Using the first cell from the recorded data as a test case, to check output.

# y_train = np.array([76,38,45,18,42,29,6,40,46,37,39,41,32,33,32,15,21,14,14,9,21,30,44,10,8,24,34,63,17,17,45,47,24,3,
# 0,44,31,3,21,3,6,21,95,80,33,50,81,39,47,30,8,5,13,38,19,26,36,6,17,28,31,22,39,54,53,33,28,32,29,19,
# 33,44,37,35,11,44,54,61,60,39,45,33,18,2,31,66,69,52,53,46,42,11,13,18,5,0,0,0,9,51,18,7,0,29,62,47,41,47,44,
# 50,28,80,56,72,72,72,56,46,53,55,74,69,69,69,71,79,89,88,68,70,58,35,25,18,25,34,15,29,28,16,40,47,23,20,23,
# 26,32,11,3,0,3,21,20,10,0,0,50,71,81,51,6,7,85,50,45,79,62,28,42,41,40,58,26,60,48,53,44,46,37,24,21,17,47,
# 34,21,12,29,10,33,13,24,68,60,75,76,69,72,79,85,84,95,71,41,83,71,25,25,13,35,17,2,2,0,10,0,24,78,65,48,37,
# 65,59,84,74,57,75,55,77,65,65,48,62,45,60,46,41,23,7,33,47])
# y_test = np.array([30,46,75,55,71,60,53,42,52,35,20,33,12,48,64,65,70,60,79,63,39,30,49,75,25,1,3,2,26,39,6,0,0,
# 17,109,90,61,85,61,43,50,53,64,66,47,59,44,69,45,58,42,55,64,61,50,41,36,73,60,49])

# x_train = np.arange(0,y_train.size,1)*5
# x_test = np.arange(y_train.size,y_test.size + y_train.size,1)*5

# data = {"NEUTRAL": (np.array(y_train[np.newaxis,:]),np.array(y_test[np.newaxis,:]),x_train.flatten(),x_test.flatten())}


In [ ]:

# Fitting scale:
scale = torch.tensor(10.0)

softplus = torch.nn.Softplus(beta=1,threshold=20)
link_function = nonlinearity(softplus)

In [ ]:
def plot_training_results(train_x, train_y, test_x, test_y,
                          latent_test_func_quantiles, latent_train_func_quantiles,
                          cell_name, index,quantiles_with_noise,quantiles_with_noise_training,
                          fontsize=12):
    """
    Plots the training results for a cell over test and training data.

    Parameters:
    - train_x, train_y: training data
    - test_x, test_y: test data
    - latent_percentiles: (lower, mean, upper) for test predictions
    - latent_training_percentiles: (lower, mean, upper) for train predictions
    - y_sim: sample from the model (unused now)
    - cell_name, index: identifiers for the plot (unused here)
    - fontsize: unified font size for labels and legend
    """
    lower, mean, upper = latent_test_func_quantiles
    lower_train, mean_train, upper_train = latent_train_func_quantiles
    lower_poisson_test, mean_poisson_test, upper_poisson_test = quantiles_with_noise
    lower_poisson_train, mean_poisson_train, upper_poisson_train = quantiles_with_noise_training

    fig, ax = plt.subplots(figsize=(25, 3))
    ax: Axes
    # GP predictions with uncertainty for test data
    ax.plot(test_x, mean, label='GP f(x) prediction', color='purple')
    ax.plot(test_x, mean_poisson_test, label='GP Likelihood test prediction', color='red')
    ax.plot(train_x, mean_poisson_train, label='GP Likelihood train prediction', color='blue')
    ax.fill_between(
        test_x,
        lower,
        upper,
        color='brown', alpha=0.4
    )

    # GP predictions with uncertainty for train data
    ax.plot(train_x, mean_train, label='GP train prediction', color='green')
    ax.fill_between(
        train_x,
        lower_train,
        upper_train,
        color='brown', alpha=0.4
    )

    ax.fill_between(
        test_x,
        lower_poisson_test,
        upper_poisson_test,
        color='gray', alpha=0.2
    )

    ax.fill_between(
        train_x,
        lower_poisson_train,
        upper_poisson_train,
        color='gray', alpha=0.2
    )

    # True functions
    ax.scatter(test_x, test_y, label='True test function', alpha=0.5, color='black')
    ax.scatter(train_x, train_y, label='True train data', alpha=0.5, color=get_cell_colour(names=cell_name))

    # Style adjustments
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.tick_params(labelsize=fontsize)
    ax.legend(fontsize=fontsize)
    ax.set_xlabel("Time(s)", fontsize=fontsize)
    ax.set_ylabel("Spike Count", fontsize=fontsize)
    ax.grid(False)

    plt.show()

In [ ]:
def fit_all_cells_and_return_params_fits(data, base_save_dir, num_iter,
                                         num_inducing, from_loaded=True,
                                         save_params=False,save_results=False,from_last=True,max_count=100,lr=0.1,cell_ids: list=None,
                                         compare_mean=True,model_mean=False,retrain_mean=False):
    """
    Fit a Poisson GP model for each cell and save R², key params, and full model param store.
    """

    # Create save directory and structures
    os.makedirs(f"{base_save_dir}/saved_models", exist_ok=True)
    summary_list = []
    loss_histories = {}


    # Iterate over each type of cell (ON, OFF, NEUTRAL)
    for (cell_name, cell_data) in data.items():
        train_ys, test_ys, train_x, test_x = cell_data
        print(train_ys.shape)
        print(test_ys.shape)
        print(train_x.shape)
        print(test_x.shape)
        test_x = torch.from_numpy(test_x).float()
        train_x = torch.from_numpy(train_x).float()
        T_max = torch.max(train_x)

        # Iterate over each individual cell
        for i, (train_y, test_y) in enumerate(zip(train_ys, test_ys)):
            model_id = f"{cell_name}_{i}"

            if cell_ids is not None:
                if model_id not in cell_ids:
                    continue

            print(model_id)
            model_param_path = os.path.join(base_save_dir, "saved_models", f"{model_id}_model_params.pt")
            mean_model_param_path = os.path.join(base_save_dir, "saved_models", f"{model_id}_mean_model_params.pt")
            lik_param_path = os.path.join(base_save_dir, "saved_models", f"{model_id}_likelihood_params.pt")
            mean_lik_param_path = os.path.join(base_save_dir, "saved_models", f"{model_id}_mean_likelihood_params.pt")

            if from_last:
                if os.path.exists(
                    os.path.join(base_save_dir,"saved_models",f"{model_id}_model_params.pt")
                    ):
                    continue

            
            train_y = torch.from_numpy(train_y).float()
            test_y = torch.from_numpy(test_y).float()

            model, likelihood, mll = create_model(T_min = 0 ,T_max=T_max, max_count=max_count,
                                                  link_function=link_function,link_function_scale=scale,
                                                  num_data = train_y.size(0),num_inducing=num_inducing,
                                                  model_mean=model_mean)
            

            print("alpha_init: ",likelihood.alpha.item(),"output_scale_init: ", model.main_scale)
            set_mean_and_scale(model,likelihood,scale,train_y,has_scale=True)


            # Train the model if not loading
            if from_loaded:
                if not os.path.exists(model_param_path):
                    raise FileNotFoundError(f"Saved model not found at {model_param_path}")
                model.load_state_dict(torch.load(model_param_path))
                likelihood.load_state_dict(torch.load(lik_param_path))
            else:
                
                loss_history = train_model(model,likelihood,mll,train_x,train_y,training_iterations=num_iter,lr=lr,mean_model=False)
                loss_histories[model_id] = loss_history

                save_model_params(model,likelihood,model_param_path,lik_param_path,save=save_params)
            


            latent_percents,quantiles_with_noise,latent_percents_train,quantiles_with_noise_training = evaluate_model(
                model,link_function,scale,likelihood,test_x,train_x,test_y,train_y,i,cell_name=cell_name,model_mean=model_mean)

            print(train_y.shape)
            print(test_y.shape)
            print(train_x.shape)
            print(test_x.shape)
            plot_training_results(
                train_x, train_y, test_x, test_y,
                latent_percents,
                latent_percents_train,
                cell_name,i, quantiles_with_noise
                ,quantiles_with_noise_training,fontsize=12,
            )

            results=None
            if compare_mean:
                # alpha_emp = 1.0 - torch.sqrt(torch.mean(train_y)/torch.var(train_y))
                #create the mean model
                mean_model, mean_lik, mean_mll = create_mean_model(T_max,max_count=train_y.max(),
                                                                   link_function=link_function,link_function_scale=scale,
                                                                   num_data=train_y.size(0),
                                                                   num_inducing=num_inducing, model_mean=model_mean)
                set_mean_and_scale(mean_model,mean_lik,scale,train_y,has_scale=False)
                if from_loaded and not retrain_mean:
                    mean_model.load_state_dict(torch.load(mean_model_param_path))
                    mean_lik.load_state_dict(torch.load(mean_lik_param_path),strict=False)
                else:
                    mean_loss_history = train_model(mean_model,mean_lik,mean_mll,train_x,train_y,training_iterations=num_iter,lr=lr,mean_model=True)
                    save_model_params(mean_model,mean_lik,mean_model_param_path,mean_lik_param_path,save=save_params)
                mean_function_results = evaluate_model(mean_model,link_function,scale,mean_lik,test_x,train_x,test_y,train_y,i,cell_name=cell_name,plot=True,model_mean=model_mean)

                #! Could plot mean function here
            

                results = compare_log_liks(model,likelihood,mean_model,mean_lik,train_x,train_y,test_x,test_y)
                
            summary_list.append(get_summary_parameters_and_fits(model,likelihood,model_id,cell_name,i,results))


    # Save summary CSV
    if save_results:
        summary_df = pd.DataFrame(summary_list)
        if not from_loaded:
            with open(os.path.join(base_save_dir, "loss_histories.csv"),'w') as f:
                json.dump(loss_histories,f)
        summary_df.to_csv(os.path.join(base_save_dir, "model_summary_sim_neuts.csv"), index=False)
        print(f"Saved {len(summary_list)} model results to {base_save_dir}")

In [ ]:
data = generate_data()


In [ ]:

fit_all_cells_and_return_params_fits(data, "./results/sim_neuts", num_iter=500,
                                        from_loaded=False, num_inducing =100,
                                            save_params=True,save_results=True,from_last=True,
                                            max_count=200,lr=0.05,compare_mean=True,cell_ids=None,model_mean=True,
                                            retrain_mean=False)